# Historical notebook — archived protocol

Outputs below predate the audited pipeline and are not new experiment results. Retained for reading. Use the staged CLI in ../README.md for new experiments. Full execution is deliberately disabled: old cells can overwrite historical artifacts and use superseded evaluation splits.

In [ ]:
raise RuntimeError("Archived notebook: use the staged commands in README.md. Historical execution is disabled.")

# 02 — Modeling & Evaluation

This notebook performs:
1. Data loading and validation splitting
2. ML-SMOTE for class imbalance
3. Binary Relevance training using XGBoost and LightGBM
4. Per-label threshold tuning
5. Evaluation on test set
6. ONNX model export

In [1]:
import sys
sys.path.append('..')  # Ensure src can be imported
import json
import time
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split

from src.config import path, CONFIG
from src.resampling import ml_smote

warnings.filterwarnings('ignore')

PROC = path('data_processed')
REPORTS = Path('../reports')
MODELS = Path('../models')
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)


In [2]:
print('[1/7] Loading processed data ...')
X_train = sp.load_npz(PROC / 'X_train.npz').toarray().astype(np.float32)
X_test  = sp.load_npz(PROC / 'X_test.npz').toarray().astype(np.float32)
Y_train = sp.load_npz(PROC / 'Y_train.npz').toarray().astype(np.float32)
Y_test  = sp.load_npz(PROC / 'Y_test.npz').toarray().astype(np.float32)
labels  = json.loads((PROC / 'label_names.json').read_text())

print(f'X_train: {X_train.shape}, Y_train: {Y_train.shape}')
print(f'X_test : {X_test.shape}, Y_test : {Y_test.shape}')
print(f'Labels : {len(labels)}')

print('\n[2/7] Membagi validation set dari training (80/20) ...')
X_tr, X_val, Y_tr, Y_val = train_test_split(
    X_train, Y_train, test_size=0.2, random_state=42
)
print(f'Train : {X_tr.shape[0]} sampel')
print(f'Val   : {X_val.shape[0]} sampel')
print(f'Test  : {X_test.shape[0]} sampel (HOLD-OUT)')


[1/7] Loading processed data ...
X_train: (5569, 2048), Y_train: (5569, 111)
X_test : (1467, 2048), Y_test : (1467, 111)
Labels : 111

[2/7] Membagi validation set dari training (80/20) ...
Train : 4455 sampel
Val   : 1114 sampel
Test  : 1467 sampel (HOLD-OUT)


In [3]:
print('[3/7] Applying ML-SMOTE pada training set ...')
X_tr_res, Y_tr_res = ml_smote(X_tr, Y_tr, k=5, sampling_ratio=0.5, seed=42)


[3/7] Applying ML-SMOTE pada training set ...
  [ML-SMOTE] Sampel minoritas: 2655 / 4455


  [ML-SMOTE] Sampel sintetis dibuat: 1327
  [ML-SMOTE] Dataset final: 5782 sampel (dari 4455 asli)


In [4]:
def find_best_threshold(y_true, y_prob):
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.1, 0.9, 81):
        f1 = f1_score(y_true, (y_prob >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t

def evaluate(Y_true, Y_pred_binary, Y_pred_proba, label_names):
    return {
        'accuracy':  float(accuracy_score(Y_true, Y_pred_binary)),
        'precision': float(precision_score(Y_true, Y_pred_binary, average='macro', zero_division=0)),
        'recall':    float(recall_score(Y_true, Y_pred_binary, average='macro', zero_division=0)),
        'f1_macro':  float(f1_score(Y_true, Y_pred_binary, average='macro', zero_division=0)),
        'roc_auc':   float(roc_auc_score(Y_true, Y_pred_proba, average='macro')),
        'f1_per_label': {
            label: float(f1_score(Y_true[:, i], Y_pred_binary[:, i], zero_division=0))
            for i, label in enumerate(label_names)
        },
    }


In [5]:
from xgboost import XGBClassifier

print('[4/7] Training XGBoost Binary Relevance ...')
n_labels = Y_tr_res.shape[1]
xgb_models, xgb_thresholds = [], []
xgb_proba_val  = np.zeros((len(X_val),  n_labels))
xgb_proba_test = np.zeros((len(X_test), n_labels))

t0 = time.time()
for j, label in enumerate(labels):
    y_tr_j  = Y_tr_res[:, j]
    y_val_j = Y_val[:, j]

    n_pos = y_tr_j.sum()
    n_neg = len(y_tr_j) - n_pos
    spw = max(1.0, n_neg / max(n_pos, 1))

    clf = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        tree_method='hist',
        eval_metric='logloss',
        use_label_encoder=False,
        verbosity=0,
        random_state=42,
    )
    clf.fit(X_tr_res, y_tr_j)
    xgb_models.append(clf)

    pv = clf.predict_proba(X_val)[:, 1]
    pt = clf.predict_proba(X_test)[:, 1]
    xgb_proba_val[:, j]  = pv
    xgb_proba_test[:, j] = pt

    t_best = find_best_threshold(y_val_j, pv)
    xgb_thresholds.append(t_best)

    if (j + 1) % 20 == 0 or j == n_labels - 1:
        elapsed = time.time() - t0
        print(f'  Label {j+1}/{n_labels} selesai ({elapsed:.1f}s)')

xgb_thresholds = np.array(xgb_thresholds)
xgb_pred = (xgb_proba_test >= xgb_thresholds).astype(int)

np.save(MODELS / 'xgb_thresholds.npy', xgb_thresholds)
xgb_model_dir = MODELS / 'xgb_models'
xgb_model_dir.mkdir(exist_ok=True)
for j, (clf, label) in enumerate(zip(xgb_models, labels)):
    safe = label.replace(' ', '_').replace('/', '-')
    joblib.dump(clf, xgb_model_dir / f'xgb_{safe}.pkl')
print(f'[XGBoost] Training selesai dan model tersimpan ke {xgb_model_dir}/')


[4/7] Training XGBoost Binary Relevance ...


  Label 20/111 selesai (145.9s)


  Label 40/111 selesai (291.0s)


  Label 60/111 selesai (438.8s)


  Label 80/111 selesai (587.0s)


  Label 100/111 selesai (736.5s)


  Label 111/111 selesai (818.6s)


[XGBoost] Training selesai dan model tersimpan ke ..\models\xgb_models/


In [6]:
from lightgbm import LGBMClassifier

print('[5/7] Training LightGBM Binary Relevance ...')
n_labels = Y_tr_res.shape[1]
lgbm_models, lgbm_thresholds = [], []
lgbm_proba_val  = np.zeros((len(X_val),  n_labels))
lgbm_proba_test = np.zeros((len(X_test), n_labels))

t0 = time.time()
for j, label in enumerate(labels):
    y_tr_j  = Y_tr_res[:, j]
    y_val_j = Y_val[:, j]

    clf = LGBMClassifier(
        n_estimators=300,
        num_leaves=63,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        is_unbalance=True,
        verbosity=-1,
        random_state=42,
    )
    clf.fit(X_tr_res, y_tr_j)
    lgbm_models.append(clf)

    pv = clf.predict_proba(X_val)[:, 1]
    pt = clf.predict_proba(X_test)[:, 1]
    lgbm_proba_val[:, j]  = pv
    lgbm_proba_test[:, j] = pt

    t_best = find_best_threshold(y_val_j, pv)
    lgbm_thresholds.append(t_best)

    if (j + 1) % 20 == 0 or j == n_labels - 1:
        elapsed = time.time() - t0
        print(f'  Label {j+1}/{n_labels} selesai ({elapsed:.1f}s)')

lgbm_thresholds = np.array(lgbm_thresholds)
lgbm_pred = (lgbm_proba_test >= lgbm_thresholds).astype(int)

np.save(MODELS / 'lgbm_thresholds.npy', lgbm_thresholds)
lgbm_model_dir = MODELS / 'lgbm_models'
lgbm_model_dir.mkdir(exist_ok=True)
for j, (clf, label) in enumerate(zip(lgbm_models, labels)):
    safe = label.replace(' ', '_').replace('/', '-')
    joblib.dump(clf, lgbm_model_dir / f'lgbm_{safe}.pkl')
print(f'[LightGBM] Training selesai dan model tersimpan ke {lgbm_model_dir}/')


[5/7] Training LightGBM Binary Relevance ...


  Label 20/111 selesai (39.0s)


  Label 40/111 selesai (85.7s)


  Label 60/111 selesai (128.1s)


  Label 80/111 selesai (169.0s)


  Label 100/111 selesai (225.4s)


  Label 111/111 selesai (251.0s)


[LightGBM] Training selesai dan model tersimpan ke ..\models\lgbm_models/


In [7]:
print('[6/7] Evaluasi pada test set (HOLD-OUT) ...')
xgb_metrics  = evaluate(Y_test, xgb_pred,  xgb_proba_test,  labels)
lgbm_metrics = evaluate(Y_test, lgbm_pred, lgbm_proba_test, labels)

print('=' * 50)
print('HASIL EVALUASI')
print('=' * 50)
headers = ['Metrik', 'XGBoost', 'LightGBM']
rows = [
    ['Accuracy',  f"{xgb_metrics['accuracy']:.4f}",  f"{lgbm_metrics['accuracy']:.4f}"],
    ['Precision', f"{xgb_metrics['precision']:.4f}", f"{lgbm_metrics['precision']:.4f}"],
    ['Recall',    f"{xgb_metrics['recall']:.4f}",    f"{lgbm_metrics['recall']:.4f}"],
    ['F1-Macro',  f"{xgb_metrics['f1_macro']:.4f}",  f"{lgbm_metrics['f1_macro']:.4f}"],
    ['ROC-AUC',   f"{xgb_metrics['roc_auc']:.4f}",   f"{lgbm_metrics['roc_auc']:.4f}"],
]
print(f"{headers[0]:<12} {headers[1]:>10} {headers[2]:>10}")
print('-' * 34)
for row in rows:
    print(f"{row[0]:<12} {row[1]:>10} {row[2]:>10}")

(REPORTS / 'results_xgb.json').write_text(json.dumps(xgb_metrics, indent=2), encoding='utf-8')
(REPORTS / 'results_lgbm.json').write_text(json.dumps(lgbm_metrics, indent=2), encoding='utf-8')

comparison = pd.DataFrame({
    'Metrik':    ['Accuracy', 'Precision', 'Recall', 'F1-Macro', 'ROC-AUC'],
    'XGBoost':   [xgb_metrics[k]  for k in ['accuracy','precision','recall','f1_macro','roc_auc']],
    'LightGBM':  [lgbm_metrics[k] for k in ['accuracy','precision','recall','f1_macro','roc_auc']],
})
comparison.to_csv(REPORTS / 'comparison_table.csv', index=False)
from IPython.display import display
display(comparison)


[6/7] Evaluasi pada test set (HOLD-OUT) ...


HASIL EVALUASI
Metrik          XGBoost   LightGBM
----------------------------------
Accuracy         0.0000     0.0014
Precision        0.1801     0.1462
Recall           0.2323     0.1869
F1-Macro         0.1768     0.1499
ROC-AUC          0.6706     0.6523


,Metrik,XGBoost,LightGBM
0,Accuracy,0.000000,0.001363
1,Precision,0.180087,0.146223
2,Recall,0.232347,0.186907
3,F1-Macro,0.176756,0.149890
4,ROC-AUC,0.670631,0.652283


In [8]:
print('[7/7] Exporting ke ONNX ...')

def export_onnx_xgb(models, labels):
    from onnxmltools.convert import convert_xgboost
    from onnxmltools.convert.common.data_types import FloatTensorType
    xgb_onnx_dir = MODELS / 'xgb_onnx'
    xgb_onnx_dir.mkdir(exist_ok=True)
    for j, (clf, label) in enumerate(zip(models, labels)):
        initial_type = [('float_input', FloatTensorType([None, 2048]))]
        try:
            onnx_model = convert_xgboost(clf, initial_types=initial_type)
            fname = xgb_onnx_dir / f"xgb_{label.replace(' ', '_')}.onnx"
            fname.write_bytes(onnx_model.SerializeToString())
        except Exception as e:
            pass
    n_ok = len(list(xgb_onnx_dir.glob('*.onnx')))
    print(f'[ONNX] XGBoost: {n_ok}/{len(labels)} models saved to {xgb_onnx_dir}/')

def export_onnx_lgbm(models, labels):
    from onnxmltools.convert import convert_lightgbm
    from onnxmltools.convert.common.data_types import FloatTensorType
    lgbm_onnx_dir = MODELS / 'lgbm_onnx'
    lgbm_onnx_dir.mkdir(exist_ok=True)
    for j, (clf, label) in enumerate(zip(models, labels)):
        initial_type = [('float_input', FloatTensorType([None, 2048]))]
        try:
            onnx_model = convert_lightgbm(clf, initial_types=initial_type, zipmap=False)
            fname = lgbm_onnx_dir / f"lgbm_{label.replace(' ', '_')}.onnx"
            fname.write_bytes(onnx_model.SerializeToString())
        except Exception as e:
            pass
    n_ok = len(list(lgbm_onnx_dir.glob('*.onnx')))
    print(f'[ONNX] LightGBM: {n_ok}/{len(labels)} models saved to {lgbm_onnx_dir}/')

export_onnx_xgb(xgb_models, labels)
export_onnx_lgbm(lgbm_models, labels)


[7/7] Exporting ke ONNX ...


[ONNX] XGBoost: 111/111 models saved to ..\models\xgb_onnx/


[ONNX] LightGBM: 111/111 models saved to ..\models\lgbm_onnx/
